In [3]:
import numpy as np
import pandas as pd

In [4]:
df=pd.read_csv("powerplant_data.csv")

In [5]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [6]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [7]:
X=df.drop("PE",axis=1)
y=df["PE"]

In [8]:
X.head(2)

,AT,V,AP,RH
0,8.34,40.77,1010.84,90.01
1,23.64,58.49,1011.40,74.20


In [9]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test= train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [10]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.fit_transform(X_test)

In [11]:
X_train_scaled

array([[ 0.74805289,  0.72006931, -0.32660017, -0.49711722],
       [ 0.86181948,  1.26515721, -0.98521113,  0.8181501 ],
       [ 0.93409473,  1.52314975,  0.32523844,  0.80167494],
       ...,
       [-0.22097078, -0.834965  ,  0.36756563, -0.83554456],
       [ 0.94747903,  1.14245344, -0.41971997, -0.45455637],
       [-1.77355014, -1.19049131,  1.92520594,  0.91837402]])

In [12]:
# Converting data into Tensors

import torch
import torch.nn as nn

X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)   #for pd series .values is used

X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [13]:
X_train_tensor
##type(X_train_tensor)

tensor([[ 0.7481,  0.7201, -0.3266, -0.4971],
        [ 0.8618,  1.2652, -0.9852,  0.8182],
        [ 0.9341,  1.5231,  0.3252,  0.8017],
        ...,
        [-0.2210, -0.8350,  0.3676, -0.8355],
        [ 0.9475,  1.1425, -0.4197, -0.4546],
        [-1.7736, -1.1905,  1.9252,  0.9184]])

In [14]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)

In [15]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True);
test_loader=DataLoader(test_dataset,batch_size=32)

# Defining Model

In [19]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        
        self.model=nn.Sequential(
        #first hidden layer
        nn.Linear(X_train.shape[1],6),  #in fea,out_fea
        nn.ReLU(),

        #second hidden layes
        nn.Linear(6,6),
        nn.ReLU(),

        #output
        nn.Linear(6,1),    
    )

    def forward(self,x):
        return self.model(x)

In [20]:
import torch.optim as optim

model=ANN()

#loss, optimzer

criterion=nn.MSELoss()
optimizer=optim.Adam(model.parameters())

In [23]:
trainloss=[]
valloss=[]   #validation

best_validation_loss=float("inf")

epochs=40

for epoch in range(epochs):
    model.train()
    runningloss=0.0

    #train
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs=model(xb)  #forward prop
        loss=criterion(outputs,yb)  #calc loss
        loss.backward()  #back prop .. compute gradients
        optimizer.step() #update params
        
        runningloss+=loss.item()

    epocs_train_loss=runningloss/len(train_loader)
    trainloss.append(epocs_train_loss)

    #validate
    model.eval()
    running_val_loss=0.0

    with torch.no_grad():
        for xb,yb in test_loader:
            output=model(xb)
            loss=criterion(output,yb)
            running_val_loss+=loss

    epocs_val_loss=running_val_loss/len(test_loader)
    valloss.append(epocs_val_loss)

    if epocs_val_loss<best_validation_loss:
        best_validation_loss=epocs_val_loss
        torch.save(model.state_dict(),"best_model.pt")

    print(f"epoch ${epoch+1}/{epochs}==> train loss = {epocs_train_loss} and validation loss={epocs_val_loss}")

epoch $1/40==> train loss = 149758.78844401042 and validation loss=120592.65625
epoch $2/40==> train loss = 90983.88061523438 and validation loss=64777.0625
epoch $3/40==> train loss = 47165.81253255208 and validation loss=34543.38671875
epoch $4/40==> train loss = 27284.363330078126 and validation loss=22145.6328125
epoch $5/40==> train loss = 18875.822790527345 and validation loss=16202.626953125
epoch $6/40==> train loss = 14289.384334309896 and validation loss=12256.6552734375
epoch $7/40==> train loss = 10786.084590657552 and validation loss=9012.005859375
epoch $8/40==> train loss = 7834.013756306967 and validation loss=6354.58837890625
epoch $9/40==> train loss = 5429.6893595377605 and validation loss=4310.333984375
epoch $10/40==> train loss = 3626.3755935668946 and validation loss=2847.5634765625
epoch $11/40==> train loss = 2372.5328035990397 and validation loss=1857.4180908203125
epoch $12/40==> train loss = 1530.4633620262146 and validation loss=1194.68505859375
epoch $13/4

In [24]:
#loading the best model

model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [29]:
#evaluation
model.eval()
with torch.no_grad():
    train_pred=model(X_train_tensor)
    test_pred=model(X_test_tensor)

    train_mse_loss=criterion(train_pred,y_train_tensor)
    test_mse_loss=criterion(test_pred,y_test_tensor)

print("treaining mse=",train_mse_loss.item())
print("testing mse=", test_mse_loss.item())

treaining mse= 21.359439849853516
testing mse= 19.88288688659668


In [31]:
from sklearn.metrics import r2_score

print("r2 score=" , r2_score(y_test,test_pred))

r2 score= 0.9305144508760372
